# DuckDB + Parquet: R Tutorial

This notebook covers the same ground as the Python tutorial, entirely in R.
It uses the same synthetic dataset — no real patient data.

**What we cover:**
1. Environment check
2. First-run setup: CSV → Parquet (via arrow)
3. Querying Parquet directly with DuckDB
4. Creating a persistent DuckDB database
5. Basic queries — SQL via DBI and dplyr
6. Joining tables
7. Researcher pattern: filtered working subset

**Prerequisites:**
```r
install.packages(c('IRkernel', 'duckdb', 'dplyr', 'arrow', 'readr', 'DBI'))
IRkernel::installspec()
```
Then launch JupyterLab and select the R kernel.

## 1. Environment check

In [ ]:
library(DBI)
library(duckdb)
library(dplyr)
library(arrow)
library(readr)

cat("R:       ", R.version.string, "\n")
cat("duckdb:  ", as.character(packageVersion("duckdb")),  "\n")
cat("arrow:   ", as.character(packageVersion("arrow")),   "\n")
cat("dplyr:   ", as.character(packageVersion("dplyr")),   "\n")

## 2. First-run setup: CSV → Parquet

Run once. Reads source CSVs from `data/raw_csv/` and writes Parquet files
to `data/parquet/` using the `arrow` package.

In [ ]:
raw_dir     <- file.path("..", "..", "data", "raw_csv")
parquet_dir <- file.path("..", "..", "data", "parquet")
dir.create(parquet_dir, recursive = TRUE, showWarnings = FALSE)

csv_files <- list.files(raw_dir, pattern = "\\.csv$", full.names = TRUE)

for (f in csv_files) {
    df       <- read_csv(f, show_col_types = FALSE)
    out_name <- sub("\\.csv$", ".parquet", basename(f))
    out_path <- file.path(parquet_dir, out_name)
    write_parquet(df, out_path)
    cat(sprintf("%-30s → %s  (%d rows)\n", basename(f), out_name, nrow(df)))
}

## 3. Query Parquet directly — no database needed

DuckDB can query Parquet files in place via an in-memory connection.
Useful for quick exploration before committing to a persistent database.

In [ ]:
# In-memory connection — nothing written to disk
mem_con <- dbConnect(duckdb::duckdb())

patients_pq <- file.path(parquet_dir, "patients.parquet")

dbGetQuery(mem_con, sprintf("
    SELECT *
    FROM read_parquet('%s')
    LIMIT 5
", patients_pq))

In [ ]:
# Aggregate directly on Parquet
dbGetQuery(mem_con, sprintf("
    SELECT
        diagnosis_year,
        sex,
        COUNT(*) AS n_patients
    FROM read_parquet('%s')
    GROUP BY diagnosis_year, sex
    ORDER BY diagnosis_year, sex
", patients_pq))

In [ ]:
dbDisconnect(mem_con, shutdown = TRUE)

## 4. Create a persistent DuckDB database

In [ ]:
db_path <- file.path("..", "..", "data", "tutorial.duckdb")
con     <- dbConnect(duckdb::duckdb(), db_path)

# Load each Parquet file as a table
pq_files <- list.files(parquet_dir, pattern = "\\.parquet$", full.names = TRUE)

for (f in pq_files) {
    tname <- tools::file_path_sans_ext(basename(f))
    dbExecute(con, sprintf("
        CREATE OR REPLACE TABLE %s AS
        SELECT * FROM read_parquet('%s')
    ", tname, f))
    n <- dbGetQuery(con, sprintf("SELECT COUNT(*) AS n FROM %s", tname))$n
    cat(sprintf("  %-20s %d rows\n", tname, n))
}

cat("\nTables in database:\n")
print(dbListTables(con))

## 5. Basic queries

### 5a. SQL via DBI

In [ ]:
# Patients per diagnosis code
dbGetQuery(con, "
    SELECT
        diagnosis_code,
        COUNT(*)            AS n_patients,
        MIN(diagnosis_year) AS first_seen,
        MAX(diagnosis_year) AS last_seen
    FROM patients
    GROUP BY diagnosis_code
    ORDER BY n_patients DESC
")

In [ ]:
# Filter: Icelandic patients diagnosed 2015+
dbGetQuery(con, "
    SELECT *
    FROM patients
    WHERE country_code   = 'IS'
      AND diagnosis_year >= 2015
    ORDER BY diagnosis_year, patient_id
")

### 5b. dplyr interface — no SQL needed

`tbl()` creates a lazy reference — queries only run when `collect()` is called.

In [ ]:
patients  <- tbl(con, "patients")
visits    <- tbl(con, "visits")
medications <- tbl(con, "medications")

# Same filter as above, dplyr style
patients |>
    filter(country_code == "IS", diagnosis_year >= 2015) |>
    count(diagnosis_year, diagnosis_code, sort = TRUE) |>
    collect()

In [ ]:
# Visit type breakdown
visits |>
    count(visit_type, clinic, sort = TRUE) |>
    collect()

## 6. Joining tables

In [ ]:
# SQL join: patients with visit counts and distinct drugs
dbGetQuery(con, "
    SELECT
        p.patient_id,
        p.sex,
        p.diagnosis_code,
        p.diagnosis_year,
        COUNT(DISTINCT v.visit_id)  AS n_visits,
        COUNT(DISTINCT m.drug_name) AS n_distinct_drugs
    FROM patients p
    LEFT JOIN visits      v ON p.patient_id = v.patient_id
    LEFT JOIN medications m ON p.patient_id = m.patient_id
    GROUP BY p.patient_id, p.sex, p.diagnosis_code, p.diagnosis_year
    ORDER BY n_visits DESC
    LIMIT 10
")

In [ ]:
# dplyr join: most prescribed drugs per diagnosis
medications |>
    left_join(patients |> select(patient_id, diagnosis_code),
              by = "patient_id") |>
    count(diagnosis_code, drug_name, sort = TRUE) |>
    group_by(diagnosis_code) |>
    slice_max(n, n = 3) |>
    collect()

## 7. Researcher pattern: filtered working subset

Read from central Parquet (read-only source of truth), write a filtered
working `.duckdb` scoped to the project's needs.

In [ ]:
working_db_path <- file.path("..", "..", "data", "my_project_r.duckdb")
work_con        <- dbConnect(duckdb::duckdb(), working_db_path)

patients_pq  <- file.path(parquet_dir, "patients.parquet")
visits_pq    <- file.path(parquet_dir, "visits.parquet")

# Filtered patient subset
dbExecute(work_con, sprintf("
    CREATE OR REPLACE TABLE patients AS
    SELECT *
    FROM read_parquet('%s')
    WHERE country_code   = 'IS'
      AND diagnosis_year >= 2015
", patients_pq))

# Visits for those patients only
dbExecute(work_con, sprintf("
    CREATE OR REPLACE TABLE visits AS
    SELECT v.*
    FROM read_parquet('%s') v
    WHERE v.patient_id IN (SELECT patient_id FROM patients)
", visits_pq))

n_p <- dbGetQuery(work_con, "SELECT COUNT(*) AS n FROM patients")$n
n_v <- dbGetQuery(work_con, "SELECT COUNT(*) AS n FROM visits")$n
cat(sprintf("Working subset: %d patients, %d visits\n", n_p, n_v))
cat(sprintf("Saved to: %s\n", working_db_path))

dbDisconnect(work_con, shutdown = TRUE)

In [ ]:
# Always close the main connection when done
dbDisconnect(con, shutdown = TRUE)

## Next steps

- See the Python notebook (`notebooks/python/01_intro_duckdb_parquet.ipynb`)
  for the same workflow in Python
- The migration toolkit (separate repo) shows how legacy Excel/Access files
  get converted to the Parquet files used here